In [48]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, optimizers
from keras.models import load_model
from keras.preprocessing import image
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, f1_score

In [49]:
# set the input image size for the ResNet50 model
img_width, img_height = 299, 299

In [50]:
# set the directories for the training,validation,testing data
train_dir = 'Train_Tomatoes'
val_dir = 'Validate_Tomatoes'
test_dir = 'Test_Tomatoes'

In [51]:
# set the batch size for training and validation data
batch_size = 15

In [52]:
# create a data generator with data augmentation for the training data
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

In [53]:
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=True
)

val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)


Found 144 images belonging to 3 classes.
Found 84 images belonging to 3 classes.
Found 42 images belonging to 3 classes.


In [54]:
# load the ResNet50 model pretrained on ImageNet without the classification layers
base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=(img_width, img_height, 3))

In [55]:
# Freeze all layers in the base model
for layer in base_model.layers:
    layer.trainable = False

In [56]:
# Add custom classification head
x = layers.Flatten()(base_model.output)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dense(512, activation='relu')(x)
x = layers.Dropout(0.5)(x)
output = layers.Dense(3, activation='softmax')(x)

In [57]:
# Compile the model
model = keras.models.Model(inputs=base_model.input, outputs=output)
model.compile(loss='categorical_crossentropy',
              optimizer=optimizers.Adam(learning_rate=1e-5),
              metrics=['accuracy'])

In [ ]:
# Train the model
epochs = 30
history = model.fit(
    train_generator,
    epochs=epochs,
    validation_data=val_generator
)


Epoch 1/30
10/10 [==============================] - 32s 2s/step - loss: 1.1283 - accuracy: 0.3958 - val_loss: 0.8915 - val_accuracy: 0.5476
Epoch 2/30
10/10 [==============================] - 21s 2s/step - loss: 0.7669 - accuracy: 0.6667 - val_loss: 0.5809 - val_accuracy: 0.8214
Epoch 3/30
10/10 [==============================] - 21s 2s/step - loss: 0.5772 - accuracy: 0.7361 - val_loss: 0.4449 - val_accuracy: 0.8929
Epoch 4/30
10/10 [==============================] - 21s 2s/step - loss: 0.4353 - accuracy: 0.8681 - val_loss: 0.3807 - val_accuracy: 0.9167
Epoch 5/30
10/10 [==============================] - 20s 2s/step - loss: 0.3372 - accuracy: 0.8681 - val_loss: 0.3189 - val_accuracy: 0.9286
Epoch 6/30
10/10 [==============================] - 21s 2s/step - loss: 0.2396 - accuracy: 0.9306 - val_loss: 0.3191 - val_accuracy: 0.9167
Epoch 7/30
10/10 [==============================] - 21s 2s/step - loss: 0.1820 - accuracy: 0.9375 - val_loss: 0.2662 - val_accuracy: 0.9167
Epoch 8/30
10/10 [==

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# Make predictions on the test data
test_preds = model.predict(test_generator)

# Convert the predictions to class labels
test_pred_labels = np.argmax(test_preds, axis=1)

# Get the true class labels for the test data
test_true_labels = test_generator.classes

# Calculate precision, recall, and F1 score
import seaborn as sns
import matplotlib.pyplot as plt

# Generate confusion matrix
cm = confusion_matrix(test_true_labels, test_pred_labels)

# Define class labels
class_labels = ["raw", "ripe", "rotten"]

# Plot confusion matrix with color
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, cmap="Reds", fmt="d", xticklabels=class_labels, yticklabels=class_labels)
plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.title("Confusion Matrix")
plt.show()

# Print classification report
print(classification_report(test_true_labels, test_pred_labels))

print('F1 score:', f1_score(test_true_labels, test_pred_labels, average='weighted'))

In [ ]:
fig , ax = plt.subplots(1,2)
train_acc = history.history['accuracy']
train_loss = history.history['loss']
fig.set_size_inches(12,4)

ax[0].plot(history.history['accuracy'])
ax[0].plot(history.history['val_accuracy'])
ax[0].set_title('Training Accuracy vs Validation Accuracy')
ax[0].set_ylabel('Accuracy')
ax[0].set_xlabel('Epoch')
ax[0].legend(['Train', 'Validation'], loc='upper left')

ax[1].plot(history.history['loss'])
ax[1].plot(history.history['val_loss'])
ax[1].set_title('Training Loss vs Validation Loss')
ax[1].set_ylabel('Loss')
ax[1].set_xlabel('Epoch')
ax[1].legend(['Train', 'Validation'], loc='upper left')

plt.show()

In [ ]:
# Save the model
model.save('inceptionv3_model_old.h5')